# ABO150 QLoRA For Qwen3 (Colab)

Этот ноутбук запускает полный цикл для `dataset/abo_150_expanded`:
1. ставит зависимости;
2. клонирует публичный репозиторий;
3. подхватывает `HF_TOKEN` и `HF_USERNAME` из Colab Secrets;
4. собирает `train/val` dataset для QLoRA;
5. дообучает `qwen3_vl_8b`;
6. при желании пушит adapter на Hugging Face Hub;
7. валидирует adapter на фиксированном holdout `50` объектов.

Colab Secrets:
- `HF_TOKEN`
- `HF_USERNAME` (опционально, но удобно для автогенерации repo id)
- `comet_api_key` / `comet_workspace` / `comet_project_name` — опционально


In [ ]:
# Dependencies (Colab-safe versions + one-time auto-restart)
import os
import sys
import subprocess
from pathlib import Path

MARKER = Path('/tmp/abo150_qlora_qwen3_deps_ready')

if not MARKER.exists():
    print('Installing dependencies (first run)...')
    install_cmds = [
        [sys.executable, '-m', 'pip', 'uninstall', '-y', 'numpy'],
        [sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '--force-reinstall', 'numpy==2.1.3'],
        [sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '--upgrade', '--upgrade-strategy', 'only-if-needed',
         'transformers>=4.49.0,<5.0.0', 'accelerate', 'bitsandbytes', 'peft', 'sentencepiece', 'huggingface_hub'],
        [sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '--upgrade', '--upgrade-strategy', 'only-if-needed',
         'pandas==2.2.2', 'pillow<12', 'pyyaml', 'boto3', 'rembg', 'onnxruntime', 'comet_ml'],
    ]

    for cmd in install_cmds:
        print('>>', ' '.join(cmd))
        subprocess.run(cmd, check=True)

    MARKER.write_text('ok')
    print('Dependencies installed. Restarting runtime now...')
    os.kill(os.getpid(), 9)
else:
    print('Dependencies already installed in this runtime. Continue.')


In [ ]:
# Clone or update public repo in Colab
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/Yaitco/VLM-2D-Physics-Boundaries.git'
WORKDIR = Path('/content/VLM-2D-Physics-Boundaries')

if not WORKDIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(WORKDIR)], check=True)
else:
    print(f'Repo already exists: {WORKDIR}. Pulling latest...')
    subprocess.run(['git', '-C', str(WORKDIR), 'pull', '--ff-only'], check=True)

%cd /content/VLM-2D-Physics-Boundaries


In [ ]:
# Load secrets and export env vars
import os

def _get_secret(name: str):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    return os.getenv(name)

HF_TOKEN = _get_secret('HF_TOKEN')
HF_USERNAME = _get_secret('HF_USERNAME')

if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
if HF_USERNAME:
    os.environ['HF_USERNAME'] = HF_USERNAME

try:
    from huggingface_hub import login
    if HF_TOKEN:
        login(token=HF_TOKEN, add_to_git_credential=True)
        print('HF token configured.')
    else:
        print('HF_TOKEN is missing. You can still build dataset, but push-to-hub will fail.')
except Exception as exc:
    print('HF login skipped:', exc)

print('HF_USERNAME:', HF_USERNAME or '<missing>')


In [ ]:
# QLoRA experiment config
from pathlib import Path
import os

DATASET_NAME = 'abo_150_expanded'
PROTOCOL_NAME = 'abo150_natural_bg_v2_transfer'
BASE_MODEL_KEY = 'qwen3_vl_8b'

TRAIN_IDS_PATH = Path('dataset/abo_150_expanded/splits/seed42_val50_train100/train_ids.txt')
VAL_IDS_PATH = Path('dataset/abo_150_expanded/splits/seed42_val50_train100/val_ids.txt')

TRANSFER_PROPERTY_KEYS = [
    'material',
    'rigidity',
    'transparency',
    'surface',
    'fragility',
]

PROPERTY_GROUP_NAME = 'natural_bg_v2_transfer'
PROPERTY_KEYS = TRANSFER_PROPERTY_KEYS
OUTPUT_TAG = f'abo150_qwen3_{PROPERTY_GROUP_NAME}'
TRAINSET_DIR = Path('outputs') / f'{OUTPUT_TAG}_dataset'
ADAPTER_DIR = Path('outputs') / OUTPUT_TAG

PUSH_TO_HUB = True
HUB_MODEL_ID = None  # e.g. 'your-hf-name/abo150_qwen3_natural_bg_v2_transfer'
HUB_PRIVATE = False

NUM_TRAIN_EPOCHS = 6
LEARNING_RATE = 2e-4
GRAD_ACC_STEPS = 4
TRAIN_BATCH_SIZE = 2
EVAL_BATCH_SIZE = 2
LOGGING_STEPS = 5
SAVE_STEPS = 25
EVAL_STEPS = 25
DISABLE_TQDM = False
USE_GRADIENT_CHECKPOINTING = False

VALIDATION_VARIANTS = ['raw']

COMET_ENABLED = True
COMET_PROJECT_NAME = 'vlm-physics-training'
VALIDATION_COMET_ENABLED = True
VALIDATION_COMET_PROJECT_NAME = 'vlm-physics-validation'
CUSTOM_MODEL_KEY = f'{BASE_MODEL_KEY}_qlora_{OUTPUT_TAG}'

print('Base model:', BASE_MODEL_KEY)
print('Protocol:', PROTOCOL_NAME)
print('Property keys:', PROPERTY_KEYS)
print('Train ids:', TRAIN_IDS_PATH)
print('Val ids:', VAL_IDS_PATH)
print('Trainset dir:', TRAINSET_DIR)
print('Adapter dir:', ADAPTER_DIR)
print('Push to hub:', PUSH_TO_HUB)
if HUB_MODEL_ID:
    print('Hub model id:', HUB_MODEL_ID)
elif os.getenv('HF_USERNAME'):
    print('Hub model id (auto):', f"{os.getenv('HF_USERNAME')}/{ADAPTER_DIR.name}")
else:
    print('Hub model id: will be inferred from token if possible')


In [ ]:
# Sanity-check split files
import json

train_ids = [line.strip() for line in TRAIN_IDS_PATH.read_text(encoding='utf-8').splitlines() if line.strip()]
val_ids = [line.strip() for line in VAL_IDS_PATH.read_text(encoding='utf-8').splitlines() if line.strip()]

print('Train ids:', len(train_ids))
print('Val ids:', len(val_ids))
print('First train ids:', train_ids[:5])
print('First val ids:', val_ids[:5])


In [ ]:
# Build per-property QLoRA dataset
import subprocess

cmd = [
    'python', 'scripts/build_abo150_qlora_dataset.py',
    '--protocol-name', PROTOCOL_NAME,
    '--property-keys', ','.join(PROPERTY_KEYS),
    '--train-ids-path', str(TRAIN_IDS_PATH),
    '--val-ids-path', str(VAL_IDS_PATH),
    '--output-dir', str(TRAINSET_DIR),
]
print('>>', ' '.join(cmd))
subprocess.run(cmd, check=True)

manifest = json.loads((TRAINSET_DIR / 'manifest.json').read_text(encoding='utf-8'))
print(json.dumps(manifest, ensure_ascii=False, indent=2))


In [ ]:
# Train locally, then push only adapter artifacts to Hugging Face Hub
import importlib
import json
import os
import shutil
import sys
from pathlib import Path

from huggingface_hub import create_repo, upload_folder

from scripts.abo150_vlm_validation import MODEL_REGISTRY
import scripts.train_abo150_qlora as train_qlora

importlib.reload(train_qlora)

if PUSH_TO_HUB and not os.getenv('HF_TOKEN'):
    raise ValueError('HF_TOKEN is missing. Load HF secrets first or set PUSH_TO_HUB = False.')
if PUSH_TO_HUB and not (HUB_MODEL_ID or os.getenv('HF_USERNAME')):
    raise ValueError('HF_USERNAME is missing. Set HUB_MODEL_ID explicitly or load HF_USERNAME secret.')

train_argv = [
    'train_abo150_qlora.py',
    '--train-jsonl', str(TRAINSET_DIR / 'train.jsonl'),
    '--val-jsonl', str(TRAINSET_DIR / 'val.jsonl'),
    '--model-key', BASE_MODEL_KEY,
    '--output-dir', str(ADAPTER_DIR),
    '--num-train-epochs', str(NUM_TRAIN_EPOCHS),
    '--learning-rate', str(LEARNING_RATE),
    '--gradient-accumulation-steps', str(GRAD_ACC_STEPS),
    '--per-device-train-batch-size', str(TRAIN_BATCH_SIZE),
    '--per-device-eval-batch-size', str(EVAL_BATCH_SIZE),
    '--logging-steps', str(LOGGING_STEPS),
    '--save-steps', str(SAVE_STEPS),
    '--eval-steps', str(EVAL_STEPS),
]
if COMET_ENABLED:
    train_argv += ['--comet-project-name', COMET_PROJECT_NAME]
else:
    train_argv += ['--disable-comet']
if DISABLE_TQDM:
    train_argv += ['--disable-tqdm']
if not USE_GRADIENT_CHECKPOINTING:
    train_argv += ['--disable-gradient-checkpointing']

print('>>', ' '.join(train_argv))
old_argv = sys.argv[:]
sys.argv = train_argv
try:
    train_qlora.main()
finally:
    sys.argv = old_argv

local_manifest_path = ADAPTER_DIR / 'manifest.json'
local_runtime_cfg_path = ADAPTER_DIR / 'runtime_model_config.json'
print('Local manifest:', local_manifest_path)
print('Local runtime config:', local_runtime_cfg_path)

if PUSH_TO_HUB:
    repo_id = HUB_MODEL_ID or f"{os.environ['HF_USERNAME']}/{ADAPTER_DIR.name}"
    export_dir = ADAPTER_DIR / '_hub_export'
    if export_dir.exists():
        shutil.rmtree(export_dir)
    export_dir.mkdir(parents=True, exist_ok=True)

    for name in ['adapter_model.safetensors', 'adapter_config.json', 'README.md']:
        src = ADAPTER_DIR / name
        if src.exists():
            shutil.copy2(src, export_dir / name)

    create_repo(repo_id=repo_id, token=os.environ['HF_TOKEN'], exist_ok=True, private=HUB_PRIVATE)
    upload_folder(
        repo_id=repo_id,
        folder_path=str(export_dir),
        token=os.environ['HF_TOKEN'],
        commit_message=f'Upload QLoRA adapter: {ADAPTER_DIR.name}',
    )

    hub_runtime_cfg = {
        'backend': 'hf_chat',
        'model_id': f"{MODEL_REGISTRY[BASE_MODEL_KEY]['model_id']}+qlora@{repo_id}",
        'base_model_id': MODEL_REGISTRY[BASE_MODEL_KEY]['model_id'],
        'adapter_path': repo_id,
        'use_4bit': True,
        'max_new_tokens': int(MODEL_REGISTRY[BASE_MODEL_KEY].get('max_new_tokens', 128)),
        'do_sample': bool(MODEL_REGISTRY[BASE_MODEL_KEY].get('do_sample', False)),
        'max_prompt_batch_size': MODEL_REGISTRY[BASE_MODEL_KEY].get('max_prompt_batch_size'),
    }
    hub_runtime_cfg_path = ADAPTER_DIR / 'runtime_model_config.hub.json'
    hub_runtime_cfg_path.write_text(json.dumps(hub_runtime_cfg, ensure_ascii=False, indent=2), encoding='utf-8')

    if local_manifest_path.exists():
        manifest = json.loads(local_manifest_path.read_text(encoding='utf-8'))
        manifest['push_to_hub'] = True
        manifest['hub_model_id'] = repo_id
        local_manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

    print('Pushed adapter to:', repo_id)
    print('Uploaded files:', sorted(p.name for p in export_dir.iterdir()))
    print('Hub runtime config:', hub_runtime_cfg_path)
else:
    print('PUSH_TO_HUB = False, skipped Hub upload.')


In [ ]:
# Validate the adapter on the fixed 50-object holdout using the same pipeline as Unified
import json
import subprocess
from pathlib import Path

import pandas as pd

from scripts.abo150_vlm_validation import (
    MODEL_REGISTRY,
    filter_property_specs,
    get_available_variants,
    get_dataset_context,
    init_comet_experiment,
    is_known_value,
    load_protocol_property_specs,
    load_samples_for_dataset,
    resolve_protocol_schema_path,
    run_validation,
    save_report,
)

DATASET = get_dataset_context(DATASET_NAME)
REPORTS_DIR = DATASET.reports_dir or Path('reports')
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

if DATASET_NAME == 'abo_150_expanded':
    subprocess.run([
        'python', 'scripts/update_abo150_panel_paths.py',
        '--annotations-path', str(DATASET.annotations_path),
        '--panels-dir', str(DATASET.dataset_dir / 'selected_150_photos' / 'panels'),
        '--path-mode', 'relative',
    ], check=True)

schema_path = resolve_protocol_schema_path(DATASET, PROTOCOL_NAME)
property_specs = load_protocol_property_specs(
    protocol_name=PROTOCOL_NAME,
    schema_path=schema_path,
)

property_manifest_path = TRAINSET_DIR / 'manifest.json'
manifest_cfg = json.loads(property_manifest_path.read_text(encoding='utf-8'))
manifest_property_keys = manifest_cfg.get('property_keys')
if not isinstance(manifest_property_keys, list) or not manifest_property_keys:
    raise ValueError(f'Expected non-empty property_keys in {property_manifest_path}')
property_specs = filter_property_specs(property_specs, [str(key) for key in manifest_property_keys])

samples = load_samples_for_dataset(
    dataset=DATASET,
    property_specs=property_specs,
    protocol_name=PROTOCOL_NAME,
    sample_ids_path=VAL_IDS_PATH,
)

base_model_cfg = MODEL_REGISTRY[BASE_MODEL_KEY]
hub_cfg_path = ADAPTER_DIR / 'runtime_model_config.hub.json'
local_cfg_path = ADAPTER_DIR / 'runtime_model_config.json'
resolved_hub_model_id = HUB_MODEL_ID
if not resolved_hub_model_id and PUSH_TO_HUB and os.getenv('HF_USERNAME'):
    resolved_hub_model_id = f"{os.getenv('HF_USERNAME')}/{ADAPTER_DIR.name}"

if hub_cfg_path.exists():
    adapter_runtime_cfg = json.loads(hub_cfg_path.read_text(encoding='utf-8'))
    adapter_source = str(hub_cfg_path)
elif local_cfg_path.exists():
    adapter_runtime_cfg = json.loads(local_cfg_path.read_text(encoding='utf-8'))
    adapter_source = str(local_cfg_path)
elif resolved_hub_model_id:
    adapter_runtime_cfg = {
        'backend': 'hf_chat',
        'model_id': f"{base_model_cfg['model_id']}+qlora@{resolved_hub_model_id}",
        'base_model_id': base_model_cfg['model_id'],
        'adapter_path': resolved_hub_model_id,
        'use_4bit': True,
        'max_new_tokens': int(base_model_cfg.get('max_new_tokens', 128)),
        'do_sample': bool(base_model_cfg.get('do_sample', False)),
        'max_prompt_batch_size': base_model_cfg.get('max_prompt_batch_size'),
    }
    adapter_source = f'hub:{resolved_hub_model_id}'
else:
    raise FileNotFoundError(
        'Adapter runtime config not found locally, and hub repo id could not be inferred. '
        'Set HUB_MODEL_ID or HF_USERNAME, or run the training cell first.'
    )

ACTIVE_MODEL_REGISTRY = dict(MODEL_REGISTRY)
ACTIVE_MODEL_REGISTRY[CUSTOM_MODEL_KEY] = adapter_runtime_cfg

available_variants = get_available_variants(VALIDATION_VARIANTS, samples)
print('Adapter source:', adapter_source)
print('Loaded samples:', len(samples))
print('Selected property keys:', list(property_specs.keys()))
print('Running variants:', available_variants)
print('First sample:')
print(json.dumps(
    {
        'image_id': samples[0]['image_id'],
        'path': samples[0]['path'],
        'known_properties': sum(
            1 for k, v in samples[0]['gt_properties'].items()
            if is_known_value(property_specs[k], v)
        ),
        'available_gt_keys': [
            k for k, v in samples[0]['gt_properties'].items()
            if is_known_value(property_specs[k], v)
        ],
    },
    ensure_ascii=False,
    indent=2,
))

run_params = {
    'dataset_name': DATASET_NAME,
    'dataset_dir': str(DATASET.dataset_dir),
    'protocol_name': PROTOCOL_NAME,
    'selected_model_key': CUSTOM_MODEL_KEY,
    'selected_model_id': ACTIVE_MODEL_REGISTRY[CUSTOM_MODEL_KEY]['model_id'],
    'eval_variants_requested': ','.join(VALIDATION_VARIANTS),
    'eval_variants_actual': ','.join(available_variants),
    'sample_ids_path': str(VAL_IDS_PATH),
    'selected_property_keys': '|'.join(property_specs.keys()),
    'property_keys_manifest_path': str(property_manifest_path),
    'mask_field': 'mask_path',
    'mask_preview_field': None,
    'mask_background_mode': 'black',
    'max_samples': None,
    'random_seed': 42,
    'include_only_gt_known': False,
    'property_batch_size': 8,
    'few_shot_k': 0,
    'few_shot_selection_mode': 'fixed',
    'json_success_threshold': 1.0,
    'image_id_success_threshold': None,
    'num_properties': len(property_specs),
}

comet_experiment = init_comet_experiment(
    run_tag=f'{CUSTOM_MODEL_KEY}_{DATASET_NAME}_{PROTOCOL_NAME}',
    run_params=run_params,
    enabled=VALIDATION_COMET_ENABLED,
    default_project=VALIDATION_COMET_PROJECT_NAME,
)

try:
    variant_metrics = []

    for variant in available_variants:
        df_variant = run_validation(
            model_key=CUSTOM_MODEL_KEY,
            samples=samples,
            property_specs=property_specs,
            model_registry=ACTIVE_MODEL_REGISTRY,
            variant=variant,
            property_batch_size=8,
            include_only_gt_known=False,
            few_shot_k=0,
            few_shot_selection_mode='fixed',
            mask_background_mode='black',
            save_raw_output=True,
            json_success_threshold=1.0,
            image_id_success_threshold=None,
        )

        pm_variant, summary_variant = save_report(
            df=df_variant,
            model_key=CUSTOM_MODEL_KEY,
            variant=variant,
            model_registry=ACTIVE_MODEL_REGISTRY,
            property_specs=property_specs,
            reports_dir=REPORTS_DIR / PROTOCOL_NAME,
            comet_experiment=comet_experiment,
        )

        pm_variant = pm_variant.copy()
        pm_variant['variant'] = variant
        variant_metrics.append(pm_variant)

    if variant_metrics:
        all_variant_metrics_df = pd.concat(variant_metrics, ignore_index=True)
        display(all_variant_metrics_df)

        if set(available_variants) >= {'raw', 'masked'}:
            pivot = all_variant_metrics_df.pivot(index='property', columns='variant', values='coverage_on_gt_known_pct')
            if 'raw' in pivot.columns and 'masked' in pivot.columns:
                pivot['delta_masked_minus_raw'] = pivot['masked'] - pivot['raw']
            print('\nCoverage comparison (masked vs raw):')
            display(pivot)
finally:
    if comet_experiment is not None:
        try:
            comet_experiment.end()
        except Exception:
            pass


In [ ]:
# Optional: inspect the main result files
from pathlib import Path
import pandas as pd

run_dir = Path('reports_abo150_expanded') / PROTOCOL_NAME / CUSTOM_MODEL_KEY / VALIDATION_VARIANTS[0]
display(pd.read_csv(run_dir / 'property_metrics.csv').head(20))
display(pd.read_csv(run_dir / 'per_sample_predictions.csv').head(5))
